# Beca 18 RAG Chatbot
**Retrieval-Augmented Generation sobre el Reglamento Oficial | PRONABEC 2026**

Pipeline que responde preguntas sobre Beca 18 exclusivamente desde el documento oficial, rechazando consultas fuera de tema.

In [1]:
# Run only in Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(
        ['pip', 'install', '-q',
         'pypdf', 'tiktoken', 'langchain-text-splitters',
         'google-genai', 'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv'],
        check=True,
    )
    print('Dependencies installed.')

---
## Step 0 | Environment Setup & Gemini Client

In [2]:
import os
import time
import random
from pathlib import Path

from dotenv import load_dotenv

NL = chr(10)  # newline — avoids \n in string literals throughout the notebook

load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise EnvironmentError(
        'GEMINI_API_KEY not found. '
        'Create a .env file with: GEMINI_API_KEY=your_key_here'
    )

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
print('Gemini client initialized.')

# Paths — works both locally (notebooks/) and in Colab
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
CHROMA_PATH  = str(PROJECT_ROOT / 'chroma_db_beca18')

# Accept any PDF in data/ (filename may vary by download source)
pdf_candidates = list(DATA_DIR.glob('*.pdf'))
assert pdf_candidates, 'No PDF found in data/. Download the Beca 18 regulation PDF and place it there.'
PDF_PATH = pdf_candidates[0]

print(f'PDF path  : {PDF_PATH}')
print(f'ChromaDB  : {CHROMA_PATH}')
print('PDF found.')

Gemini client initialized.
PDF path  : C:\Users\51950\Documents\GitHub\HW_3\beca18-rag-chatbot\data\7778068-rde-n-033-2026-minedu-vmgi-pronabec.pdf
ChromaDB  : C:\Users\51950\Documents\GitHub\HW_3\beca18-rag-chatbot\chroma_db_beca18
PDF found.


---
## Step 1 | PDF Extraction with `[PAGE N]` Markers
Extract text page-by-page with `pypdf`, clean whitespace, and report character/word counts.

In [3]:
from pypdf import PdfReader


def clean_page_text(text):
    """Strip redundant whitespace and lone page-number lines."""
    lines = [line.strip() for line in text.splitlines()]
    lines = [l for l in lines if l and not l.isdigit()]
    return NL.join(lines)


reader      = PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f'Total pages: {total_pages}')

pages_data      = []  # list of {page: int, text: str}
full_text_parts = []  # for global stats

for i, page in enumerate(reader.pages):
    raw     = page.extract_text() or ''
    cleaned = clean_page_text(raw)
    pages_data.append({'page': i + 1, 'text': cleaned})
    full_text_parts.append('[PAGE ' + str(i + 1) + ']' + NL + cleaned)

full_text = (NL + NL).join(full_text_parts)

print(f'Character count : {len(full_text):,}')
print(f'Word count      : {len(full_text.split()):,}')
print(NL + '--- Preview (first 500 chars) ---')
print(full_text[:500])

Total pages: 138


Character count : 366,341
Word count      : 55,035

--- Preview (first 500 chars) ---
[PAGE 1]
Resolución Directoral Ejecutiva
Nº 033-2026-MINEDU/VMGI-PRONABEC
Lima, 24 de febrero de 2026
VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por
la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y
Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de
Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ
de la Oficina de Asesoría Jurídica, y;
CONSIDERANDO:
Que, la Ley N° 29837 crea el Prog


---
## Step 2 | Token Counting, Chunking Justification & Splitting
Count tokens with `tiktoken` (cl100k_base), justify 400-token chunks / 60-token overlap, split with metadata.

In [4]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm.notebook import tqdm as tqdm_nb

enc          = tiktoken.get_encoding('cl100k_base')
total_tokens = len(enc.encode(full_text))

print(f'Total tokens (cl100k_base) : {total_tokens:,}')
print(f'Embedding model limit      : 8,192 tokens')
print(f'Chunk size chosen          : 400 tokens  ({400 / 8192 * 100:.1f}% of limit)')
print(f'Overlap                    : 60 tokens   (maintains context at boundaries)')
print(f'Estimated num chunks       : ~{total_tokens // (400 - 60)}')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    length_function=lambda t: len(enc.encode(t)),
    separators=[chr(10) + chr(10), chr(10), '. ', ' ', ''],
)

all_chunks = []
for page_data in tqdm_nb(pages_data, desc='Chunking pages'):
    if not page_data['text'].strip():
        continue
    page_chunks = splitter.create_documents(
        texts=[page_data['text']],
        metadatas=[{
            'document': 'beca18_reglamento',
            'topic':    'beca18',
            'language': 'es',
            'page':     page_data['page'],
        }],
    )
    all_chunks.extend(page_chunks)

print(f'Total chunks generated : {len(all_chunks)}')
print(f'Sample metadata        : {all_chunks[0].metadata}')
print(f'Sample content         :\n{all_chunks[0].page_content[:250]}')

Total tokens (cl100k_base) : 108,521
Embedding model limit      : 8,192 tokens
Chunk size chosen          : 400 tokens  (4.9% of limit)
Overlap                    : 60 tokens   (maintains context at boundaries)
Estimated num chunks       : ~319


Chunking pages:   0%|          | 0/138 [00:00<?, ?it/s]

Total chunks generated : 367
Sample metadata        : {'document': 'beca18_reglamento', 'topic': 'beca18', 'language': 'es', 'page': 1}
Sample content         :
Resolución Directoral Ejecutiva
Nº 033-2026-MINEDU/VMGI-PRONABEC
Lima, 24 de febrero de 2026
VISTOS:
El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por
la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional 


---
## Step 3 | Embedding Functions with Exponential Backoff
`embed_documents()` uses `RETRIEVAL_DOCUMENT`; `embed_query()` uses `RETRIEVAL_QUERY`. Both handle rate limits (~60 req/min free tier).

In [5]:
def embed_documents(texts):
    """Embed a list of document texts using RETRIEVAL_DOCUMENT task type."""
    embeddings = []
    for text in tqdm_nb(texts, desc='Embedding docs'):
        for attempt in range(5):
            try:
                resp = client.models.embed_content(
                    model='gemini-embedding-001',
                    contents=text,
                    config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT'),
                )
                embeddings.append(list(resp.embeddings[0].values))
                break
            except Exception as exc:
                msg = str(exc).lower()
                if '429' in msg or 'quota' in msg or 'rate' in msg:
                    wait = (2 ** attempt) + random.uniform(0.1, 0.5)
                    print(f'  Rate limit (attempt {attempt + 1}/5), waiting {wait:.1f}s...')
                    time.sleep(wait)
                else:
                    raise
        time.sleep(1.1)  # stay safely under 60 req/min
    return embeddings


def embed_query(text):
    """Embed a single query string using RETRIEVAL_QUERY task type."""
    for attempt in range(5):
        try:
            resp = client.models.embed_content(
                model='gemini-embedding-001',
                contents=text,
                config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY'),
            )
            return list(resp.embeddings[0].values)
        except Exception as exc:
            msg = str(exc).lower()
            if '429' in msg or 'quota' in msg or 'rate' in msg:
                wait = (2 ** attempt) + random.uniform(0.1, 0.5)
                print(f'Rate limit (attempt {attempt + 1}/5), waiting {wait:.1f}s...')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('embed_query: all 5 retry attempts failed.')


# Smoke test
test_vec = embed_query('prueba de embedding')
print(f'Embedding dimension : {len(test_vec)}')
print(f'First 5 values      : {[round(v, 6) for v in test_vec[:5]]}')

Embedding dimension : 3072
First 5 values      : [-0.008664, -0.003855, 0.008015, -0.090736, 0.006105]


---
## Step 4 | Persistent ChromaDB Index (Idempotent)
Create or reuse a persistent ChromaDB collection (cosine distance). Skip embedding if documents already exist.

In [6]:
import chromadb

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name='beca18',
    metadata={'hnsw:space': 'cosine'},
)

if collection.count() > 0:
    print(f'Collection already has {collection.count()} chunks. Skipping indexing.')
else:
    print(f'Indexing {len(all_chunks)} chunks — may take several minutes on the free tier...')

    texts = [c.page_content for c in all_chunks]
    metas = [c.metadata     for c in all_chunks]
    ids   = ['chunk_' + str(i) for i in range(len(all_chunks))]

    embeddings = embed_documents(texts)

    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metas,
        ids=ids,
    )
    print(f'Done. {collection.count()} chunks indexed.')

Indexing 367 chunks — may take several minutes on the free tier...


Embedding docs:   0%|          | 0/367 [00:00<?, ?it/s]

Done. 367 chunks indexed.


---
## Step 5 | `semantic_search` Function
Retrieve the k most relevant chunks via cosine similarity. Test with a sample query.

In [7]:
def semantic_search(question, k=5):
    """Return top-k chunks most relevant to question."""
    q_emb   = embed_query(question)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {'text': doc, 'metadata': meta, 'distance': dist}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0],
        )
    ]


SAMPLE = '¿Cuáles son los requisitos para postular a Beca 18?'
hits   = semantic_search(SAMPLE, k=5)
print('Query: ' + SAMPLE)
print('Retrieved: ' + str(len(hits)) + ' chunks' + NL)
for i, h in enumerate(hits[:3], 1):
    page = h['metadata'].get('page')
    dist = h['distance']
    print(f'--- Result {i} | Page {page} | dist={dist:.4f} ---')
    print(h['text'][:280])
    print()

Query: ¿Cuáles son los requisitos para postular a Beca 18?
Retrieved: 5 chunks

--- Result 1 | Page 102 | dist=0.1924 ---
N° REQUISITO  DOCUMENTO DE ACREDITACIÓN o FORMA
DE ACREDITACIÓN
documento oficial de la IES donde se acredite
que mantiene dicha vacante.
Haber culminado el nivel
secundario de la Educación
Básica Regular (EBR) o
Alternativa (EBA) o Especial
(EBE). Los estudios deben ser
reconoci

--- Result 2 | Page 103 | dist=0.2002 ---
Resolución de reconocimiento de los estudios
realizados en el extranjero del Ministerio de
Educación y Certificado de estudios del colegio
de origen que contenga tod as las notas de los
estudios reconocidos.
Acreditar alto rendimiento
académico:
Para la Beca 18 Ordinaria,
Beca Hu

--- Result 3 | Page 40 | dist=0.2112 ---
7. Población Objetivo de Beca 18 y Becas Especiales
La población objetivo de Beca 18 Ordinaria está conformada por los jóvenes egresados
de la educación secundaria con tercio superior 7 y en situación de pobreza o pobreza
extrema d

---
## Step 6 | `answer_with_context` + 6 Test Questions
Grounded answers via `gemini-2.5-flash` with strict system prompt. 5 on-topic + 1 off-topic.

In [8]:
SYSTEM_PROMPT = NL.join([
    'Eres un asistente especializado en el reglamento del programa Beca 18 de PRONABEC (Peru).',
    '',
    'REGLAS ESTRICTAS:',
    '1. Responde UNICAMENTE basandote en los fragmentos de contexto proporcionados. No uses conocimiento externo.',
    '2. Cita siempre el numero de pagina de donde proviene la informacion (ej: Segun la pagina 5...).',
    '3. Si la respuesta no esta en el contexto, di exactamente: No encontre informacion sobre eso en el reglamento de Beca 18.',
    '4. Si la pregunta NO esta relacionada con Beca 18, di exactamente: Esta pregunta esta fuera del ambito del reglamento de Beca 18.',
    '5. Responde siempre en espanol.',
])


def answer_with_context(question, k=5):
    """Retrieve context chunks and generate a grounded answer via gemini-2.5-flash."""
    sources = semantic_search(question, k=k)

    context_blocks = []
    for i, s in enumerate(sources, 1):
        page  = s['metadata'].get('page', '?')
        text  = s['text']
        block = '[Fragmento ' + str(i) + ' | Pagina ' + str(page) + ']' + NL + text
        context_blocks.append(block)
    context = (NL + NL).join(context_blocks)

    user_prompt = (
        'Contexto del reglamento de Beca 18:' + NL + NL
        + context + NL + NL
        + 'Pregunta: ' + question
    )

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.1,
        ),
        contents=user_prompt,
    )
    return {'answer': response.text, 'sources': sources}


# 6 test questions
TEST_CASES = [
    ('Elegibilidad', '¿Quienes son elegibles para postular a Beca 18?'),
    ('Modalidades',  '¿Cuales son las modalidades de la beca?'),
    ('Estipendio',   '¿Cual es el monto del estipendio mensual del becario?'),
    ('Obligaciones', '¿Cuales son las obligaciones del becario durante el programa?'),
    ('Perdida',      '¿Bajo que condiciones se pierde la beca?'),
    ('Off-topic',    '¿Cual es la capital de Francia?'),
]

SEP = '=' * 65
for label, question in TEST_CASES:
    print(SEP)
    print('  [' + label + ']  ' + question)
    print(SEP)
    result = answer_with_context(question, k=5)
    print(result['answer'])
    print()

  [Elegibilidad]  ¿Quienes son elegibles para postular a Beca 18?


Para postular a Beca 18 (ordinaria), los jóvenes elegibles deben cumplir con los siguientes requisitos:

*   Ser egresados de la educación secundaria con alto rendimiento académico y en situación de vulnerabilidad o situación especial (Según la página 3).
*   Específicamente para Beca 18 Ordinaria, la población objetivo está conformada por jóvenes egresados de la educación secundaria con tercio superior y en situación de pobreza o pobreza extrema, de acuerdo con los criterios de focalización establecidos por el Sistema de Focalización de Hogares (SISFOH) (Según la página 40).

Los requisitos específicos son:
*   **Edad:** Menor de 22 años a la fecha de la publicación de la Norma Técnica (Según la página 40).
*   **Condición de vulnerabilidad:** Clasificación como pobre o pobre extremo según SISFOH (Según la página 40).
*   **Rendimiento académico:** Tercio superior en los dos últimos grados concluidos de secundaria EBR o EBE o su equivalente en EBA, según corresponda (Según la página 4

Las modalidades de la beca son las siguientes:

*   Beca 18 (ordinaria) (Según la página 89)
*   Beca de Formación en Educación Intercultural Bilingüe (Beca EIB) (Según la página 89)
*   Beca para adolescentes con protección estatal (Beca Protección) (Según la página 89)
*   Beca para Comunidades Nativas Amazónicas (Beca CNA) (Según la página 89)
*   Beca para licenciados del Servicio Militar Voluntario (Beca FF.AA.) (Según la página 89)
*   Beca para pobladores residentes del valle de los ríos Apurímac, Ene y Mantaro (Beca VRAEM) (Según la página 89)
*   Beca para pobladores residentes en el Huallaga (Beca Huallaga) (Según la página 89)
*   Beca para Pueblo Afroperuano (Beca PA) (Según la página 89)
*   Beca para víctimas de la violencia habida en el país durante los años 1980 – 2000 (Beca REPARED) (Según la página 89)
*   Beca de Excelencia Académica para Hijos de Docentes (BEAHD) (Según la página 89)

  [Estipendio]  ¿Cual es el monto del estipendio mensual del becario?


No encontre informacion sobre eso en el reglamento de Beca 18.

  [Obligaciones]  ¿Cuales son las obligaciones del becario durante el programa?


Las obligaciones del becario durante el programa son las siguientes:

*   Los derechos y obligaciones de los becarios se rigen de acuerdo con la Ley N° 29837 y su Reglamento y a las normas internas del Programa. (Según la página 122)
*   Las obligaciones académicas de los becarios están establecidas en el Reglamento y la normativa vigente del PRONABEC. (Según la página 94)
*   Cumplir con el “Compromiso del Servicio al Perú”, a fin de revertir a favor del país los beneficios de la capacitación recibida por el Estado (Anexo N° 08). (Según la página 122)
*   Dedicarse a realizar sus estudios de educación superior durante el tiempo que perciba los beneficios de la beca, debiendo, en caso de desaprobar un curso o más, asumir las responsabilidades administrativas de acuerdo a la normatividad vigente. (Según la página 136)
*   Autorizar que el PRONABEC solicite información sobre su rendimiento y riesgo académico a la institución de educación superior, así como la información de las atencione

Según el reglamento de Beca 18, las condiciones bajo las cuales se puede perder la beca son las siguientes:

*   Si producto de la fiscalización posterior se verifica que el postulante no cumple con alguno de los requisitos, será declarado No Apto durante el concurso, y si es identificado luego de ser declarado becario se procederá conforme a lo establecido en el artículo 34.3 de la Ley del Procedimiento Administrativo General, Ley N° 27444 (Según la página 110).
*   El incumplimiento de cualquiera de las estipulaciones, condiciones y/o requisitos establecidos en las Bases, es considerado causal para declarar fuera del concurso al postulante o becario (declarándolo NO APTO) o declarar la nulidad de la adjudicación de la beca (Según la página 123).
*   Haber falseado la información socioeconómica y/o académica para hacerse acreedor o continuar con una beca (Según la página 111).
*   En caso de desaprobar un curso o más, el becario debe asumir las responsabilidades administrativas de acu

Esta pregunta esta fuera del ambito del reglamento de Beca 18.



---
## Step 7 | Interactive Chat UI (`ipywidgets`)
Text input · Ask/Clear buttons · k slider · response area · collapsible source accordion.

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_input = widgets.Text(
    placeholder='Escribe tu pregunta sobre Beca 18...',
    layout=widgets.Layout(width='65%'),
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Chunks (k):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='32%'),
)
ask_button   = widgets.Button(description='Ask',   button_style='primary', icon='search')
clear_button = widgets.Button(description='Clear', button_style='warning', icon='times')
output_area  = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='12px', min_height='80px')
)


def on_ask(b):
    question = question_input.value.strip()
    with output_area:
        clear_output(wait=True)
        if not question:
            print('Por favor escribe una pregunta.')
            return
        display(widgets.HTML('<i>Consultando: ' + question + '</i>'))
        result      = answer_with_context(question, k=k_slider.value)
        answer_html = result['answer'].replace(chr(10), '<br>')
        display(widgets.HTML('<br><b>Respuesta:</b><br>' + answer_html))

        source_boxes = []
        for s in result['sources']:
            page   = s['metadata'].get('page', '?')
            dist   = s['distance']
            text   = s['text']
            header = '<b>Pagina ' + str(page) + ' | dist coseno: ' + f'{dist:.4f}' + '</b>'
            body   = '<pre style="white-space:pre-wrap;font-size:11px;background:#f8f8f8;padding:6px">' + text + '</pre>'
            source_boxes.append(widgets.HTML(header + body))

        n   = len(result['sources'])
        acc = widgets.Accordion(children=[widgets.VBox(source_boxes)])
        acc.set_title(0, 'Fuentes (' + str(n) + ' chunks recuperados)')
        acc.selected_index = None
        display(acc)


def on_clear(b):
    question_input.value = ''
    with output_area:
        clear_output()


ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

title = widgets.HTML(
    '<h3 style="margin-bottom:4px">Chatbot Beca 18 — Sistema RAG</h3>'
    '<p style="color:#555;margin-top:0">Responde exclusivamente desde el reglamento oficial de Beca 18 (PRONABEC 2026).</p>'
)
ui = widgets.VBox([
    title,
    widgets.HBox([question_input, ask_button, clear_button]),
    k_slider,
    output_area,
])
display(ui)